# 02 - Depth Anything 3 Initialization and Gaussian Splatting

Welcome back! In the previous notebook, we used **COLMAP** to estimate camera poses and a sparse 3D point cloud from image correspondences. That is the classical structure-from-motion route: find matching visual features across images, triangulate them, and use the resulting sparse geometry to initialize 3D Gaussian Splatting.

In this notebook, we try a different starting point. Instead of relying on sparse feature matches, we use **Depth Anything 3 Streaming (DA3-Streaming)** to predict a dense depth map and a camera trajectory. The depth maps are then backprojected into 3D points, which gives us a much denser initialization for Gaussian Splatting.

The comparison is intentionally narrow and controlled:

- notebook 01: sparse COLMAP points from the selected frame subset,
- this notebook: dense DA3 backprojected points from the same selected frame subset.

The goal is not to prove that one method is universally better. Instead, we want to build intuition for what changes when the 3DGS optimizer starts from **sparse feature-based geometry** versus **dense learned geometry**.

By the end of this notebook, you should be able to:

- explain the difference between sparse COLMAP geometry and dense depth-based geometry,
- inspect DA3 depth and confidence maps before trusting them,
- understand how a depth image can be backprojected into a 3D point cloud,
- train a 3D Gaussian Splatting model from the DA3 initialization,
- render a camera trajectory as a GIF,
- evaluate held camera views with PSNR and SSIM,
- reason about when dense learned depth helps and when it can introduce plausible-looking but uncertain geometry.

A useful mindset for this notebook is: **DA3 gives us more geometry, but more geometry does not automatically mean more correct geometry.**


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/FelTris/camp2_3drecon.git"
REPO_NAME = "camp2_3drecon"


def running_in_colab():
    try:
        import google.colab  # noqa: F401
    except ModuleNotFoundError:
        return False
    return True


if running_in_colab():
    repo = Path("/content") / REPO_NAME
    if not (repo / ".git").exists():
        if repo.exists():
            raise FileExistsError(f"{repo} exists but is not a Git checkout. Remove it or choose another path.")
        subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
    os.chdir(repo)
else:
    repo = Path.cwd()
    if repo.name == "notebooks":
        repo = repo.parent

REPO = repo.resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

mpl_config = REPO / "outputs" / ".mplconfig"
mpl_config.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_config))

print("Repository:", REPO)

In [ ]:
import shutil

import numpy as np
import pandas as pd
import pycolmap
import torch
from IPython.display import Image as DisplayImage
from PIL import Image
from torch.utils.data import DataLoader

from camp3d.data import ScaredSubset
from camp3d.metrics import image_metrics
from camp3d.splats import (
    GaussianSplatModel,
    PosedImage,
    PosedImageDataset,
    interpolate_camera_path,
    make_gaussian_optimizers,
    points_from_depth_prediction,
    render_posed_image,
    render_virtual_camera,
    train_one_epoch,
)
from camp3d.viz import save_render_gif, show_colmap_reconstruction_plotly, show_render_comparison

device = "cuda" if torch.cuda.is_available() else "cpu"
dataset = ScaredSubset(REPO / "scared_640")
all_image_paths = dataset.left_images()
da3_experiment = "all_frames"
experiment_strides = {
    "all_frames": 1,
    "every_5th": 5,
    "every_10th": 10,
}
image_stride = experiment_strides[da3_experiment]
image_paths = all_image_paths[::image_stride]
print("device:", device)
print("DA3 experiment:", da3_experiment)
print("frames:", len(image_paths), "of", len(all_image_paths))

## Load The Sparse COLMAP Reference

Before looking at the DA3 reconstruction, we first load the sparse COLMAP reconstruction from notebook 01. This gives us a reference point for comparison.

COLMAP reconstructs 3D points only where it can find and match visual features across multiple images. That usually means textured, well-observed areas are reconstructed more reliably, while smooth, reflective, blurry, or low-texture regions may be missing.

DA3 behaves differently. It predicts depth densely for every frame, including regions where COLMAP may not have enough feature matches. This can give much better coverage, but it also means some surfaces may be inferred by the model rather than directly supported by multi-view matching evidence.

The sparse reference is loaded from the matching experiment folder. For example, if:

```python
da3_experiment = "every_5th"
```

then the COLMAP reference comes from:

```text
outputs/01_colmap/every_5th/sparse
```

The comparison here is mainly about:

- point density,
- spatial coverage,
- camera trajectory behavior,
- and how the different initialization affects 3DGS training.

The COLMAP visualization is normalized into a unit cube to make it easier to navigate interactively. This normalization is only for plotting. It does not change the original reconstruction or the coordinates used for training.


In [ ]:
def image_camtoworld(image):
    cam_from_world_attr = getattr(image, "cam_from_world")
    cam_from_world = cam_from_world_attr() if callable(cam_from_world_attr) else cam_from_world_attr
    world_to_cam = np.eye(4, dtype=np.float32)
    world_to_cam[:3, :4] = np.asarray(cam_from_world.matrix(), dtype=np.float32)
    return np.linalg.inv(world_to_cam)


def load_colmap_reference():
    candidates = [
        REPO / "outputs" / "01_colmap" / da3_experiment / "sparse",
    ]
    if da3_experiment == "all_frames":
        candidates.append(REPO / "outputs" / "01_colmap" / "sparse")
    sparse_dir = next((path for path in candidates if path.exists()), None)
    if sparse_dir is None:
        raise FileNotFoundError(
            "No COLMAP reconstruction found. Run notebook 01 through the COLMAP section first."
        )
    model_dirs = sorted(path for path in sparse_dir.iterdir() if path.is_dir())
    if not model_dirs:
        raise FileNotFoundError(f"No sparse model directories found in {sparse_dir}.")
    model_dir, reconstruction = max(
        ((path, pycolmap.Reconstruction(path)) for path in model_dirs),
        key=lambda item: sum(image.has_pose for image in item[1].images.values()),
    )
    points = []
    colors = []
    camtoworlds = []
    image_names = []
    for point in reconstruction.points3D.values():
        points.append(point.xyz)
        colors.append(np.asarray(point.color) / 255.0)
    for image in reconstruction.images.values():
        if image.has_pose:
            camtoworlds.append(image_camtoworld(image))
            image_names.append(image.name)
    return {
        "model_dir": model_dir,
        "points": np.asarray(points, dtype=np.float32),
        "colors": np.asarray(colors, dtype=np.float32),
        "camtoworlds": np.asarray(camtoworlds, dtype=np.float32),
        "image_names": image_names,
    }


colmap_ref = load_colmap_reference()
print("COLMAP model:", colmap_ref["model_dir"])
print("COLMAP registered cameras:", len(colmap_ref["camtoworlds"]))
print("COLMAP sparse points:", len(colmap_ref["points"]))

## Prepare DA3 Streaming Input

DA3-Streaming expects a folder of input frames. This cell prepares that folder from the selected image subset.

The notebook uses a small cache mechanism so that we do not unnecessarily copy files or rerun DA3 every time the notebook is opened. The `input_manifest.txt` file stores the list of images that should be present for the current experiment. If the selected frame subset changes, the manifest no longer matches and the input folder is rebuilt.

This is useful because DA3 inference can be relatively expensive compared with simply loading already computed outputs.

Set:

```python
force_rerun_da3 = True
```

when you intentionally want to delete the cached DA3 inputs and outputs and run everything again from scratch.

In normal use, keep it as:

```python
force_rerun_da3 = False
```

so the notebook reuses existing results whenever possible.


In [ ]:
stream_root = REPO / "outputs" / "02_da3_streaming" / da3_experiment
stream_input = stream_root / "input_frames"
stream_output = stream_root / "streaming_result"
processed_dir = stream_root / "processed_images"
manifest_path = stream_root / "input_manifest.txt"
force_rerun_da3 = False

expected_manifest = [path.name for path in image_paths]
if force_rerun_da3 or not manifest_path.exists() or manifest_path.read_text().splitlines() != expected_manifest:
    for path in (stream_input, stream_output, processed_dir):
        if path.exists():
            shutil.rmtree(path)
        path.mkdir(parents=True, exist_ok=True)
    for idx, src in enumerate(image_paths):
        shutil.copy2(src, stream_input / f"frame_{idx:05d}.png")
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text("\n".join(expected_manifest) + "\n")
else:
    stream_input.mkdir(parents=True, exist_ok=True)
    stream_output.mkdir(parents=True, exist_ok=True)
    processed_dir.mkdir(parents=True, exist_ok=True)

print("stream input:", stream_input)

## Run Or Reuse DA3-Streaming

This section either loads cached DA3 outputs or runs DA3-Streaming if the required files are missing.

DA3 predicts three pieces of information that are important for us:

1. **Depth maps**  
   For each pixel, DA3 estimates how far the observed surface is from the camera.

2. **Confidence maps**  
   DA3 also provides a confidence value. This is not ground truth, but it is a useful hint about where the model thinks its prediction is more or less reliable.

3. **Camera trajectory**  
   DA3-Streaming estimates camera poses across the image sequence. These poses tell us where each camera is located and how it is oriented in 3D space.

We keep loop closure disabled here to make the output easier to inspect and reason about. Loop closure can be useful in larger SLAM-style systems, but it also adds another correction step. For teaching purposes, it is clearer to first look at the direct streaming output.

The important thing to remember is that DA3 is a learned model. It can produce dense, visually plausible geometry even in places where classical feature matching would struggle. That is powerful, but it also means we should inspect the result before using it as an initialization.


In [ ]:
poses_file = stream_output / "camera_poses.txt"
intrinsics_file = stream_output / "intrinsic.txt"
results_dir = stream_output / "results_output"

if poses_file.exists() and intrinsics_file.exists() and any(results_dir.glob("frame_*.npz")):
    print("Using cached DA3 outputs:", stream_output)
else:
    from huggingface_hub import snapshot_download

    DA3_STREAMING_DIR = REPO / "externals" / "Depth-Anything-3" / "da3_streaming"
    if str(DA3_STREAMING_DIR) not in sys.path:
        sys.path.insert(0, str(DA3_STREAMING_DIR))

    from da3_streaming import DA3_Streaming
    from loop_utils.config_utils import load_config
    from loop_utils.sim3utils import merge_ply_files

    weights_dir = Path(
        snapshot_download(
            "depth-anything/DA3-SMALL",
            allow_patterns=["config.json", "model.safetensors"],
        )
    )

    stream_config = load_config(str(DA3_STREAMING_DIR / "configs" / "base_config.yaml"))
    stream_config["Weights"]["DA3"] = str(weights_dir / "model.safetensors")
    stream_config["Weights"]["DA3_CONFIG"] = str(weights_dir / "config.json")
    stream_config["Model"]["chunk_size"] = 16
    stream_config["Model"]["overlap"] = 8
    stream_config["Model"]["loop_enable"] = False
    stream_config["Model"]["align_lib"] = "torch"
    stream_config["Model"]["delete_temp_files"] = True
    stream_config["Model"]["save_depth_conf_result"] = True
    stream_config["Model"]["save_debug_info"] = False
    stream_config["Model"]["Pointcloud_Save"]["sample_ratio"] = 0.02

    streamer = DA3_Streaming(str(stream_input), str(stream_output), stream_config)
    print(type(streamer.model))
    streamer.run()
    streamer.close()
    merge_ply_files(str(stream_output / "pcd"), str(stream_output / "pcd" / "combined_pcd.ply"))
    del streamer
    torch.cuda.empty_cache()

print("poses:", poses_file)
print("intrinsics:", intrinsics_file)

## Load And Inspect DA3 Geometry

Now we load the DA3 outputs from disk.

Each result file contains the image, the predicted depth map, and the confidence map for one frame. We also load the camera poses and camera intrinsics estimated by DA3.

The camera intrinsics describe the internal camera model, such as focal length and principal point. The camera pose describes where the camera is in the world. Together with the depth image, these are enough to convert pixels into 3D points.

The visualization below shows three views of the same frame:

- the RGB image,
- the DA3 depth map,
- the DA3 confidence map.

When inspecting this plot, try to ask:

- Do nearby and faraway structures make sense in the depth map?
- Are boundaries sharp or smeared?
- Are low-confidence regions aligned with difficult image regions, such as blur, reflections, specular highlights, textureless areas, or occlusions?
- Does the confidence map suggest that we should filter out some pixels before building the point cloud?

Confidence is not the same as correctness. A high-confidence prediction can still be wrong, and a low-confidence prediction can still be useful. But confidence is a practical signal for filtering noisy geometry before initializing 3DGS.


In [ ]:
def frame_index(path):
    return int(path.stem.split("_")[-1])


npz_paths = sorted((stream_output / "results_output").glob("frame_*.npz"), key=frame_index)
indices = [frame_index(path) for path in npz_paths]

images = []
depths = []
confs = []
processed_dir.mkdir(parents=True, exist_ok=True)
for path in npz_paths:
    data = np.load(path)
    image = data["image"]
    images.append(image)
    depths.append(data["depth"])
    confs.append(data["conf"])
    Image.fromarray(image).save(processed_dir / f"frame_{frame_index(path):05d}.png")

processed_images = np.stack(images)
depth = np.stack(depths).astype(np.float32)
confidence = np.stack(confs).astype(np.float32)

camtoworld = np.loadtxt(stream_output / "camera_poses.txt", dtype=np.float32).reshape(-1, 4, 4)
intrinsic_rows = np.loadtxt(stream_output / "intrinsic.txt", dtype=np.float32)
intrinsics = np.zeros((len(intrinsic_rows), 3, 3), dtype=np.float32)
intrinsics[:, 0, 0] = intrinsic_rows[:, 0]
intrinsics[:, 1, 1] = intrinsic_rows[:, 1]
intrinsics[:, 0, 2] = intrinsic_rows[:, 2]
intrinsics[:, 1, 2] = intrinsic_rows[:, 3]
intrinsics[:, 2, 2] = 1.0

camtoworld = camtoworld[indices]
intrinsics = intrinsics[indices]
extrinsics = np.linalg.inv(camtoworld)[:, :3, :].astype(np.float32)

print("images:", processed_images.shape)
print("depth:", depth.shape)
print("confidence:", confidence.shape)
print("camtoworld:", camtoworld.shape)
print("intrinsics:", intrinsics.shape)

In [ ]:
import matplotlib.pyplot as plt

i = len(processed_images) // 2
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(processed_images[i])
axes[0].set_title("image")
axes[1].imshow(depth[i], cmap="magma")
axes[1].set_title("DA3 depth")
axes[2].imshow(confidence[i], cmap="viridis")
axes[2].set_title("confidence")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## Backproject DA3 Depth

A depth map is still a 2D image: every pixel stores a distance value. To use it for 3D Gaussian Splatting, we need to turn it into 3D points.

This is done by **backprojection**. Conceptually, each pixel defines a ray leaving the camera. The depth value tells us how far along that ray the surface point should be. Using the camera intrinsics and the camera pose, we can place that point in the global 3D coordinate system.

Because DA3 predicts depth densely, a single frame can produce many points. Across many frames, this can quickly become a very large point cloud. We therefore apply a few practical filters:

- `confidence_percentile` removes the least confident depth predictions,
- `stride` keeps only every N-th pixel to reduce point count,
- `max_points` caps the final number of points.

These parameters are intentionally easy to change. Try making the point cloud denser or sparser and observe what happens.

Useful questions to keep in mind:

- Does a higher confidence threshold remove obvious noise?
- Does a smaller stride improve coverage or mostly add redundant points?
- Does the DA3 point cloud cover surfaces that COLMAP missed?
- Are some dense regions actually artifacts caused by incorrect depth?

The table compares the sparse COLMAP reference with the dense DA3 backprojection. The number of points alone is not a quality metric, but it helps show how different the two initializations are.


In [ ]:
confidence_percentile = 25
stride = 8
max_points = 80_000

points, colors = points_from_depth_prediction(
    images=processed_images,
    depths=depth,
    intrinsics=intrinsics,
    extrinsics=extrinsics,
    confidences=confidence,
    confidence_percentile=confidence_percentile,
    stride=stride,
    max_points=max_points,
)

comparison = pd.DataFrame(
    [
        {"method": "COLMAP sparse", "cameras": len(colmap_ref["camtoworlds"]), "points": len(colmap_ref["points"])},
        {"method": "DA3 backprojected", "cameras": len(camtoworld), "points": len(points)},
    ]
)
display(comparison)

In [ ]:
fig = show_colmap_reconstruction_plotly(
    colmap_ref["points"],
    colmap_ref["colors"],
    camtoworlds=colmap_ref["camtoworlds"],
    image_names=colmap_ref["image_names"],
    max_points=50_000,
    point_size=2.0,
    title=f"COLMAP sparse points from notebook 01 ({da3_experiment})",
    normalize=True,
)
fig.show()

In [ ]:
image_names = [f"frame_{idx:05d}.png" for idx in indices]
fig = show_colmap_reconstruction_plotly(
    points,
    colors,
    camtoworlds=camtoworld,
    image_names=image_names,
    max_points=80_000,
    point_size=2.0,
    title=f"DA3 dense initialization points ({da3_experiment})",
)
fig.show()

## Train 3DGS From DA3 Initialization

Now we use the DA3 backprojected point cloud to initialize a 3D Gaussian Splatting model.

A 3DGS model represents the scene as many small, learnable Gaussian blobs in 3D space. Each Gaussian has properties such as position, color, scale, opacity, and shape. During training, the model renders the scene from known camera views and updates the Gaussians so that the rendered images match the real input images.

The initialization matters because 3DGS optimization is not magic. If the starting points are close to the real scene geometry, training has an easier job. If the starting points are sparse, noisy, or in the wrong locations, the optimizer may need more time or may converge to a worse solution.

The hypothesis in this notebook is:

> A denser DA3 initialization may give 3DGS a better starting point than sparse COLMAP points, especially in areas where COLMAP has little coverage.

But there is an important caveat:

> DA3 depth is learned and can hallucinate plausible surfaces. Dense does not always mean correct.

So when looking at the final renders, we should not only ask whether they look sharp. We should also ask whether the reconstructed geometry is consistent and trustworthy.


In [ ]:
frames = [
    PosedImage(
        image_path=processed_dir / f"frame_{idx:05d}.png",
        camtoworld=camtoworld[i],
        K=intrinsics[i].astype(np.float32),
    )
    for i, idx in enumerate(indices)
]

trainset = PosedImageDataset(frames, image_scale=0.25)
trainloader = DataLoader(trainset, batch_size=1, shuffle=True)
model = GaussianSplatModel(points, colors, init_scale=0.01, init_opacity=0.1).to(device)
optimizer = make_gaussian_optimizers(model, lr=1e-3)

for epoch in range(30):
    loss = train_one_epoch(model, trainloader, optimizer, device=device)
    print(f"epoch {epoch:02d} | L1 loss {loss:.4f}")

## Rendered Views, PSNR, And SSIM

After training, we render a few known camera views and compare the rendered images against the corresponding real images.

This gives us a simple quantitative check of reconstruction quality.

We use two common image metrics:

- **PSNR** measures pixel-wise reconstruction error. Higher PSNR usually means the rendered image is numerically closer to the target image.
- **SSIM** measures similarity of local image structure. It is often more aligned with perceptual image quality than raw pixel error, but it is still only a simple metric.

Both metrics are useful, but neither metric truly understands the scene. In this notebook, that matters because surgical/endoscopic data can contain difficult visual effects such as specular highlights, blur, non-Lambertian tissue appearance, deformation, smoke, fluid, and tool motion.

So treat PSNR and SSIM as helpful summaries, not as the final truth.

A good evaluation combines:

- the metric table,
- side-by-side visual comparisons,
- inspection of the point cloud,
- and common sense about whether the geometry is actually plausible.

When comparing COLMAP and DA3 initializations, look for differences such as:

- sharper or blurrier renders,
- better or worse coverage of low-texture regions,
- floating artifacts,
- missing surfaces,
- and whether the model overfits known views without producing convincing novel views.


In [ ]:
source_sample_ids = np.linspace(0, len(image_paths) - 1, num=min(3, len(image_paths)), dtype=int)
deterministic_sample_names = [image_paths[idx].name for idx in source_sample_ids]
requested_names = deterministic_sample_names
print(f"Using deterministic {da3_experiment} source samples:", requested_names)

source_names = [image_paths[idx].name for idx in indices]
source_to_frame_id = {name: frame_id for frame_id, name in enumerate(source_names)}
matched_ids = [source_to_frame_id[name] for name in requested_names if name in source_to_frame_id]
if matched_ids:
    sample_ids = np.asarray(matched_ids, dtype=int)
else:
    print("Requested source samples were not available in DA3 outputs; using evenly spaced DA3 frames.")
    sample_ids = np.linspace(0, len(frames) - 1, num=min(3, len(frames)), dtype=int)
metric_rows = []

for frame_id in sample_ids:
    source_name = image_paths[indices[frame_id]].name
    print("rendering", source_name)
    rendered = render_posed_image(model, frames[frame_id], image_scale=0.25, device=device)
    metrics = image_metrics(rendered["render"], rendered["target"])
    metric_rows.append({"frame": frames[frame_id].image_path.name, "source_frame": source_name, **metrics})
    show_render_comparison(
        rendered["target"],
        rendered["render"],
        rendered["alpha"],
        metrics=metrics,
    )

display(pd.DataFrame(metric_rows))

## Render An Interpolated DA3 Trajectory GIF

Finally, we render a short GIF along an interpolated camera trajectory.

Instead of rendering only the original training views, we create virtual cameras between selected anchor frames. This is useful because novel-view rendering often reveals problems that are not obvious from training-view metrics.

For example, a reconstruction may look acceptable from the input cameras but fall apart when viewed from slightly different positions. Floating geometry, holes, incorrect depth ordering, and over-smoothed surfaces are often easier to spot in a moving render.

This section uses the same source-frame anchors as notebook 01, but mapped onto the DA3 camera trajectory. The coordinate systems of COLMAP and DA3 are not necessarily identical, so we do not compare raw coordinates directly. Instead, we compare the semantic camera path: we interpolate between the same original image choices and render the in-between views.

When watching the GIF, pay attention to:

- whether motion feels smooth,
- whether structures remain stable,
- whether surfaces shimmer or float,
- whether geometry disappears at certain viewpoints,
- and whether DA3 initialization improves coverage compared with the sparse COLMAP baseline.

The GIF is not just for presentation. It is a very useful debugging tool.


In [ ]:
source_sample_ids = np.linspace(0, len(image_paths) - 1, num=min(3, len(image_paths)), dtype=int)
anchor_names = [image_paths[idx].name for idx in source_sample_ids]
print(f"Using deterministic {da3_experiment} trajectory anchors:", anchor_names)

source_names = [image_paths[idx].name for idx in indices]
source_to_frame_id = {name: frame_id for frame_id, name in enumerate(source_names)}
anchor_source_names = [name for name in anchor_names if name in source_to_frame_id]
anchor_frames = [frames[source_to_frame_id[name]] for name in anchor_source_names]
if len(anchor_frames) < 2:
    raise ValueError("Need at least two DA3 anchor frames for an interpolated trajectory.")

novel_cameras = interpolate_camera_path(anchor_frames, steps_per_segment=20, include_keyframes=False)
gif_renders = [
    render_virtual_camera(
        model,
        camera["camtoworld"],
        camera["K"],
        reference_frame=anchor_frames[0],
        image_scale=1.0,
        device=device,
    )["render"]
    for camera in novel_cameras
]
gif_path = save_render_gif(stream_root / "da3_3dgs_trajectory.gif", gif_renders, fps=8)
print("anchor source frames:", anchor_source_names)
print("novel views:", len(gif_renders))
DisplayImage(filename=str(gif_path))

## Discussion

Use this section to reflect on what changed when we replaced sparse COLMAP initialization with dense DA3 initialization.

Some guiding questions:

1. **Where does DA3 provide geometry that sparse COLMAP missed?**  
   Look especially at smooth, weakly textured, blurry, or repeatedly moving regions. These are often hard for feature matching.

2. **Which DA3-filled surfaces look plausible but may not be directly supported by matching evidence?**  
   Dense learned depth can fill in surfaces that look reasonable, but that does not guarantee multi-view correctness.

3. **Why might DA3 renders from all frames look blurrier than the COLMAP baseline?**  
   Possible reasons include noisy depth, pose drift, inconsistent predictions between frames, too many uncertain points, or a harder optimization problem caused by dense but imperfect initialization.

4. **What changes in the `every_5th` setting?**  
   Compare COLMAP and DA3 when fewer frames are used. Does DA3 still provide dense coverage? Does COLMAP become much sparser? Does 3DGS benefit more from DA3 when the classical reconstruction has fewer matching opportunities?

5. **How do point clouds and final renders relate?**  
   A point cloud can look dense and impressive but still lead to poor renders if points are misplaced. Conversely, a sparse point cloud can sometimes be enough if the camera poses are accurate and the visible scene is simple.

6. **What would you try next?**  
   Some natural experiments are:
   - increase or decrease the confidence threshold,
   - change the point sampling stride,
   - reduce `max_points`,
   - train longer,
   - compare different frame subsets,
   - inspect failure cases visually rather than relying only on metrics.

The main takeaway is that initialization is a trade-off. COLMAP gives sparse geometry backed by feature matches. DA3 gives dense geometry from learned depth prediction. For 3DGS, the best starting point is not necessarily the densest one, but the one that gives the optimizer useful and consistent information.
